# Kalshi Earnings Markets

Pull every earnings-call market Kalshi has listed, from its public API (no login needed). These are the markets on [kalshi.com/category/mentions/earnings](https://kalshi.com/category/mentions/earnings): one yes/no contract per word, e.g. "Will AMD say *TSMC* on its next earnings call?"

Kalshi splits markets across two endpoints at a rolling cutoff, so we query both and de-duplicate:

| Endpoint | Holds |
| -------- | ----- |
| `/markets` | Open markets and recently settled ones |
| `/historical/markets` | Markets settled before the cutoff |

Output: `data/kalshi_earnings_markets.parquet`, one row per market. A full pull takes a few minutes.

## 0. Config

In [1]:
from __future__ import annotations

import re
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
import requests

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
CATEGORY = "Mentions"
EARNINGS_TAG = "Earnings"
MARKET_ENDPOINTS = ("markets", "historical/markets")
PAGE_LIMIT = 1000
MAX_WORKERS = 6
OUTPUT_PATH = ROOT / "data" / "kalshi_earnings_markets.parquet"

pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

## 1. API helpers

In [2]:
session = requests.Session()


def get_json(path: str, **params) -> dict:
    """GET one API page, backing off when rate limited or on a server error."""
    for attempt in range(8):
        response = session.get(f"{BASE_URL}/{path}", params=params, timeout=60)
        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(1 + attempt)
            continue
        response.raise_for_status()
        return response.json()
    response.raise_for_status()
    raise RuntimeError(f"gave up on {path} {params}: HTTP {response.status_code}")


def get_series_markets(series_ticker: str) -> list[dict]:
    """Every market for one series, from both the live and historical endpoints."""
    markets: list[dict] = []
    for endpoint in MARKET_ENDPOINTS:
        cursor = None
        while True:
            params = {"series_ticker": series_ticker, "limit": PAGE_LIMIT}
            if cursor:
                params["cursor"] = cursor
            page = get_json(endpoint, **params)
            for market in page.get("markets") or []:
                markets.append({**market, "series_ticker": series_ticker, "endpoint": endpoint})
            cursor = page.get("cursor")
            if not cursor:
                break
    return markets


def get_many(series_tickers: list[str]) -> list[dict]:
    with ThreadPoolExecutor(MAX_WORKERS) as pool:
        return [m for batch in pool.map(get_series_markets, series_tickers) for m in batch]


get_json("historical/cutoff")

{'market_positions_last_updated_ts': '2026-09-25T00:00:00Z',
 'market_settled_ts': '2026-09-25T00:00:00Z',
 'orders_updated_ts': '2026-11-10T00:00:00Z',
 'trades_created_ts': '2026-09-25T00:00:00Z'}

## 2. Find the earnings series

Kalshi makes one series per company. The `Earnings` tag is what the web page's filter shows. Series tickers aren't a reliable way to find them: they vary (`KXEARNINGSMENTIONAMD`, `KXMENTIONEARNWGO`, `KXBRKEM`) and one is even misspelled (`KXEARNIGNSMENTIONJPM`).

In [3]:
mentions = pd.DataFrame(get_json("series", category=CATEGORY)["series"])
is_tagged = mentions["tags"].map(lambda tags: EARNINGS_TAG in (tags if tags is not None else []))
earnings_series = mentions.loc[is_tagged, "ticker"].tolist()
other_series = mentions.loc[~is_tagged, "ticker"].tolist()

print(f"{CATEGORY} series={len(mentions)}  tagged {EARNINGS_TAG}={len(earnings_series)}  untagged={len(other_series)}")
mentions.loc[is_tagged, ["ticker", "title", "tags"]].head()

Mentions series=502  tagged Earnings=231  untagged=271


,ticker,title,tags
0,KXEARNINGSMENTIONMRK,Merck Earnings Call,[Earnings]
4,KXEARNINGSMENTIONAPH,Amphenol Earnings Call,[Earnings]
8,KXEARNINGSMENTIONUNP,Union Pacific Earnings Call,[Earnings]
10,KXEARNINGSMENTIONKLAR,Klarna Earnings Call,[Earnings]
12,KXEARNINGSMENTIONON,ON Semi,[Earnings]


## 3. Pull the markets

In [4]:
tagged_markets = pd.DataFrame(get_many(earnings_series))
tagged_markets["on_earnings_page"] = True
print(f"tagged: rows={len(tagged_markets)}")
print(tagged_markets["endpoint"].value_counts().to_string())

tagged: rows=9900
endpoint
historical/markets    5811
markets               4089


Some earnings-call markets sit in series without the tag, such as Elon Musk markets on the SpaceX call. They aren't on the web page, so they're kept but flagged `on_earnings_page = False`.

In [5]:
other_markets = pd.DataFrame(get_many(other_series))
says_earnings_call = other_markets["rules_primary"].fillna("").str.contains(r"\bearnings call\b", case=False)
untagged_markets = other_markets.loc[says_earnings_call].assign(on_earnings_page=False)

print(f"untagged series scanned: markets={len(other_markets)}  on an earnings call={len(untagged_markets)}")
untagged_markets[["ticker", "rules_primary"]].head(3)

untagged series scanned: markets=81798  on an earnings call=17


,ticker,rules_primary
71182,KXELONMENTION-26AUG04-TESL,If Elon Musk says Tesla as part of SpaceX Earn...
71183,KXELONMENTION-26AUG04-STAR,If Elon Musk says Starbase as part of SpaceX E...
71184,KXELONMENTION-26AUG04-SATE,If Elon Musk says Satellite as part of SpaceX ...


## 4. Clean up

Markets settled around the cutoff appear on both endpoints with identical values, so we keep one row per `ticker`. API numbers arrive as strings, and `word` is what must be said.

**Don't trust the date in the event ticker.** `KXEARNINGSMENTIONGOOGL-25NOV04` was really the Oct 29, 2025 call, and `-26MAR31` is sometimes a placeholder. Checked against Yahoo earnings dates for the 40 most-listed companies, the ticker date was within a day only 67% of the time. The median market close time per call (New York time) was within a day 98% of the time, so:

| Column | Meaning |
| ------ | ------- |
| `call_date` | Settled calls: median close date, accurate to about a day. Unsettled calls: the ticker date, which is only a schedule guess |
| `call_date_source` | `close_time` or `ticker` |
| `ticker_date` | Raw date from the event ticker. Blank for early events (`-25MAY` has no day) |

In [6]:
# Series whose suffix isn't the stock ticker.
TICKER_ALIASES = {
    "ADOBE": "ADBE",
    "ALIBABA": "BABA",
    "ARITZIA": "ATZ",
    "COINBASE": "COIN",
    "DOLLARGENERAL": "DG",
    "LUCID": "LCID",
}


def company_ticker(series_ticker: str) -> object:
    if series_ticker == "KXBRKEM":
        return "BRK"
    match = re.search(r"(?:MENTIONEARN|MENTION)([A-Z0-9]+)$", series_ticker)
    if not match:
        return pd.NA
    return TICKER_ALIASES.get(match.group(1), match.group(1))


def add_call_dates(frame: pd.DataFrame) -> pd.DataFrame:
    ticker_date = frame["event_ticker"].str.extract(r"-(\d{2}[A-Z]{3}\d{2})$", expand=False)
    frame["ticker_date"] = pd.to_datetime(ticker_date.str.title(), format="%y%b%d", errors="coerce")

    is_done = frame["status"].isin(["finalized", "closed"])
    median_close = frame.loc[is_done].groupby("event_ticker")["close_time"].median()
    close_date = median_close.dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()

    frame["call_date"] = frame["event_ticker"].map(close_date)
    frame["call_date_source"] = frame["call_date"].notna().map({True: "close_time", False: "ticker"})
    frame["call_date"] = frame["call_date"].fillna(frame["ticker_date"])
    return frame


def clean_markets(raw: pd.DataFrame) -> pd.DataFrame:
    frame = raw.drop_duplicates("ticker").copy()
    frame["company_ticker"] = frame["series_ticker"].map(company_ticker).astype("string")
    frame["company"] = (
        frame["title"].str.extract(r"^What will (?:representatives of )?(.+?) say", expand=False)
        .fillna(frame["rules_primary"].str.extract(r"said by any (.+?) representative", expand=False))
        .fillna(frame["rules_primary"].str.extract(r"part of (.+?) Earnings Call", expand=False))
        .str.strip()
    )
    frame["word"] = frame["custom_strike"].map(lambda s: s.get("Word") if isinstance(s, dict) else None)
    frame["word"] = frame["word"].fillna(frame["yes_sub_title"])

    for name in ["open_time", "close_time", "settlement_ts"]:
        frame[name] = pd.to_datetime(frame[name], utc=True, format="ISO8601")
    frame = add_call_dates(frame)

    numeric = {
        "volume_fp": "volume",
        "open_interest_fp": "open_interest",
        "last_price_dollars": "last_price",
        "settlement_value_dollars": "settlement_value",
    }
    for source, target in numeric.items():
        frame[target] = pd.to_numeric(frame[source], errors="coerce")

    columns = [
        "ticker", "event_ticker", "series_ticker", "company_ticker", "company", "word",
        "call_date", "call_date_source", "ticker_date",
        "status", "result", "settlement_value", "last_price", "volume", "open_interest",
        "open_time", "close_time", "settlement_ts", "on_earnings_page", "endpoint", "rules_primary",
    ]
    return frame.loc[:, columns].sort_values(["call_date", "company_ticker", "ticker"]).reset_index(drop=True)


earnings_markets = clean_markets(pd.concat([tagged_markets, untagged_markets], ignore_index=True))
assert earnings_markets["ticker"].is_unique

print(f"rows before de-duplicating={len(tagged_markets) + len(untagged_markets)}  markets={len(earnings_markets)}")
earnings_markets.head()

rows before de-duplicating=9917  markets=9862


,ticker,event_ticker,series_ticker,company_ticker,company,word,call_date,call_date_source,ticker_date,status,result,settlement_value,last_price,volume,open_interest,open_time,close_time,settlement_ts,on_earnings_page,endpoint,rules_primary
0,KXEARNINGSMENTIONNVDA-25FEB-AMAZON,KXEARNINGSMENTIONNVDA-25FEB,KXEARNINGSMENTIONNVDA,NVDA,NVIDIA Corp,Amazon,2025-02-27,close_time,NaT,finalized,no,0.0,0.01,5594.0,0.0,2025-02-26 17:00:00+00:00,2025-02-27 15:12:48.678720+00:00,2025-02-27 15:42:50.392489+00:00,True,historical/markets,If Amazon is said by any NVIDIA Corp represent...
1,KXEARNINGSMENTIONNVDA-25FEB-AMDINTEL,KXEARNINGSMENTIONNVDA-25FEB,KXEARNINGSMENTIONNVDA,NVDA,NVIDIA Corp,AMD/Intel,2025-02-27,close_time,NaT,finalized,no,0.0,0.01,1559.0,0.0,2025-02-26 17:00:00+00:00,2025-02-27 15:15:03.149445+00:00,2025-02-27 15:45:05.288241+00:00,True,historical/markets,If AMD/Intel is said by any NVIDIA Corp repres...
2,KXEARNINGSMENTIONNVDA-25FEB-APPLE,KXEARNINGSMENTIONNVDA-25FEB,KXEARNINGSMENTIONNVDA,NVDA,NVIDIA Corp,Apple,2025-02-27,close_time,NaT,finalized,no,0.0,0.01,1671.0,0.0,2025-02-26 17:00:00+00:00,2025-02-27 15:12:12.842301+00:00,2025-02-27 15:42:14.298486+00:00,True,historical/markets,If Apple is said by any NVIDIA Corp representa...
3,KXEARNINGSMENTIONNVDA-25FEB-BLACKWELL,KXEARNINGSMENTIONNVDA-25FEB,KXEARNINGSMENTIONNVDA,NVDA,NVIDIA Corp,Blackwell,2025-02-27,close_time,NaT,finalized,yes,1.0,0.99,4146.0,0.0,2025-02-26 17:00:00+00:00,2025-02-27 14:53:57.294601+00:00,2025-02-27 15:23:58.318325+00:00,True,historical/markets,If Blackwell is said by any NVIDIA Corp repres...
4,KXEARNINGSMENTIONNVDA-25FEB-BUYBACK,KXEARNINGSMENTIONNVDA-25FEB,KXEARNINGSMENTIONNVDA,NVDA,NVIDIA Corp,Buyback,2025-02-27,close_time,NaT,finalized,no,0.0,0.01,1900.0,0.0,2025-02-26 17:00:00+00:00,2025-02-27 15:13:53.136534+00:00,2025-02-27 15:43:55.288922+00:00,True,historical/markets,If Buyback is said by any NVIDIA Corp represen...


## 5. What we got

In [7]:
print(f"markets={len(earnings_markets)}  calls={earnings_markets['event_ticker'].nunique()}  series={earnings_markets['series_ticker'].nunique()}")
print(f"first opened={earnings_markets['open_time'].min():%Y-%m-%d}  last opened={earnings_markets['open_time'].max():%Y-%m-%d}")
for name in ["call_date", "company", "word"]:
    print(f"missing {name}={earnings_markets[name].isna().sum()}")
print(f"tagged series with no markets={sorted(set(earnings_series) - set(earnings_markets['series_ticker']))}")
pd.concat(
    [earnings_markets["status"].value_counts(), earnings_markets["result"].replace("", "(unsettled)").value_counts()],
    keys=["status", "result"],
).to_frame("markets")

markets=9862  calls=710  series=228
first opened=2025-02-26  last opened=2026-11-24
missing call_date=0
missing company=0
missing word=0
tagged series with no markets=['KXEARNINGSMENTIONACA', 'KXEARNINGSMENTIONATZ', 'KXEARNINGSMENTIONDEER', 'KXEARNINGSMENTIONSYUM']


markets
status finalized       7841
       active          1970
       inactive          27
       closed            24
result yes             4125
       no              3714
       (unsettled)     2021
       scalar             2

In [8]:
# Settled calls only; upcoming calls have no result and only a guessed date.
settled = earnings_markets.loc[earnings_markets["result"].isin(["yes", "no"])]
by_quarter = (
    settled.assign(
        quarter=settled["call_date"].dt.to_period("Q").astype("string"),
        said_yes=(settled["result"] == "yes").astype("float"),
    )
    .groupby("quarter")
    .agg(
        calls=("event_ticker", "nunique"),
        companies=("series_ticker", "nunique"),
        markets=("ticker", "size"),
        volume=("volume", "sum"),
        yes_rate=("said_yes", "mean"),
    )
    .round({"volume": 0, "yes_rate": 3})
)
by_quarter

,calls,companies,markets,volume,yes_rate
quarter,,,,,
2025Q1,8,8,120,368070.0,0.467
2025Q2,18,18,315,1278235.0,0.543
2025Q3,21,21,383,1945532.0,0.567
2025Q4,64,64,811,10357680.0,0.564
2026Q1,101,101,1172,33230731.0,0.576
2026Q2,93,93,1231,8972042.0,0.526
2026Q3,127,126,1884,13987866.0,0.502
2026Q4,135,135,1923,5669588.0,0.497


In [9]:
(
    earnings_markets.groupby(["company_ticker", "company"], dropna=False)
    .agg(calls=("event_ticker", "nunique"), markets=("ticker", "size"), volume=("volume", "sum"))
    .sort_values("volume", ascending=False)
    .head(15)
    .round({"volume": 0})
)

,,calls,markets,volume
company_ticker,company,,,
EA,Electronic Arts Inc.,1,13,9015889.0
TSLA,"Tesla, Inc.",4,81,2536269.0
NVDA,NVIDIA Corporation,7,148,2438771.0
MCD,McDonald's Corporation,6,103,1594178.0
NFLX,"Netflix, Inc.",6,97,1545507.0
GOOGL,Alphabet Inc.,5,82,1384497.0
COST,Costco Wholesale Corporation,4,66,1363085.0
TGT,Target Corporation,6,82,1259742.0
PLTR,Palantir Technologies Inc.,4,67,1249063.0


## 6. Overlap with our stock universe

A rough ticker match against `data/stock_universe.csv`, comparing both the names and the dates each dataset covers.

In [10]:
universe = pd.read_csv(ROOT / "data" / "stock_universe.csv")
shared = sorted(set(universe["Symbol"].dropna()) & set(earnings_markets["company_ticker"].dropna()))

print(f"universe symbols={universe['Symbol'].nunique()}  also on Kalshi={len(shared)}")
print(shared)
print(f"universe price_end max={universe['price_end'].max()}  first Kalshi call={earnings_markets['call_date'].min():%Y-%m-%d}")
print(f"settled calls before the universe ends={(settled.loc[settled['company_ticker'].isin(shared) & (settled['call_date'] <= universe['price_end'].max()), 'event_ticker'].nunique())}")

universe symbols=813  also on Kalshi=155
['AAPL', 'ABBV', 'ABNB', 'ABT', 'ACN', 'ADBE', 'ADI', 'ALLY', 'AMAT', 'AMD', 'AMGN', 'AMZN', 'ANET', 'APH', 'ARM', 'ASML', 'ASTS', 'AVGO', 'AXP', 'AZO', 'BA', 'BABA', 'BAC', 'BJ', 'BKNG', 'BLK', 'C', 'CAT', 'CCL', 'CIEN', 'CMG', 'COHR', 'COIN', 'COP', 'COST', 'CRCL', 'CRM', 'CRWD', 'CRWV', 'CSCO', 'CVNA', 'CVX', 'DAL', 'DASH', 'DE', 'DELL', 'DG', 'DHR', 'DIS', 'DKNG', 'DKS', 'DPZ', 'EBAY', 'F', 'FDX', 'FUTU', 'GE', 'GEV', 'GILD', 'GIS', 'GM', 'GME', 'GOOGL', 'GS', 'HD', 'HLT', 'HOOD', 'HPE', 'IBM', 'INTC', 'INTU', 'ISRG', 'JNJ', 'JPM', 'KKR', 'KLAC', 'KO', 'KR', 'LIN', 'LLY', 'LMT', 'LOW', 'LRCX', 'LULU', 'MA', 'MCD', 'MCO', 'MDT', 'MELI', 'META', 'MKC', 'MRK', 'MRVL', 'MS', 'MSFT', 'MSTR', 'MU', 'NBIS', 'NEE', 'NEM', 'NFLX', 'NKE', 'NVDA', 'NYT', 'ON', 'ORCL', 'PANW', 'PEP', 'PFE', 'PG', 'PGR', 'PLD', 'PLTR', 'PM', 'PSKY', 'PYPL', 'QCOM', 'RDDT', 'RIVN', 'RKLB', 'ROKU', 'ROST', 'RTX', 'SBUX', 'SCHW', 'SJM', 'SNDK', 'SNOW', 'SPOT', 'STZ', 'TFC',

## 7. Save

In [11]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
earnings_markets.to_parquet(OUTPUT_PATH, index=False)
print(f"wrote {OUTPUT_PATH}  rows={len(earnings_markets)}  cols={earnings_markets.shape[1]}")

wrote /Users/etlevn/Documents/Projects/IMP/data/kalshi_earnings_markets.parquet  rows=9862  cols=21
